In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Shared telescope geometry, the danish-based donut simulator, and figure
# helpers live in the installed `shape_vs_intensity` package (pip install -e .).
import galsim.zernike as gzk
from shape_vs_intensity import config as C
from shape_vs_intensity import sim
from shape_vs_intensity.plotutils import use_style

In [ ]:
use_style()

DIAGRAM_L = 1.25
DONUT_OUTER_PX = 66.7  # measured outer radius of a full-defocus donut, in pixels

def show_donut(ax, img):
    """Show a donut on ``ax``, cropped to match the schematic framing.

    Crops so a full-defocus ring fills the same fraction of the panel as the
    map_circles schematic ring (outer radius / half-width = 1 / DIAGRAM_L), then
    hides the axis ticks.
    """
    crop = C.NPIX // 2 - int(round(DONUT_OUTER_PX * DIAGRAM_L))
    ax.imshow(img[crop:-crop, crop:-crop], origin="lower")
    ax.set(xticks=[], yticks=[])

In [ ]:
def map_circles(A, displacement, N=5, ax=None, **kwargs):
    """Map concentric pupil circles through a displacement field.

    Parameters
    ----------
    A : float
        Scale factor applied to the displacement.  With ``zernike_displacement``
        this is the wavefront coefficient in meters -- the same number passed to
        ``sim.simulate_donut``.
    displacement : callable
        Function ``displacement(rho, theta) -> (dx, dy)`` giving the Cartesian
        ring displacement in donut-radius units per unit ``A``.
    N : int, optional
        Number of pupil circles to draw.
    ax : matplotlib.axes.Axes, optional
        Axes on which to draw the circles.
    **kwargs
        Additional keyword arguments passed to ``Axes.plot``.
    """
    kwargs = {"c": "C1", "lw": 0.5, "zorder": 10} | kwargs
    if ax is None:
        fig, ax = plt.subplots()

    theta = np.linspace(0, 2 * np.pi, 10_000)
    for r in np.linspace(C.EPS_RUBIN, 1, N):
        rho = r * np.ones_like(theta)
        x0 = rho * np.cos(theta)
        y0 = rho * np.sin(theta)
        dx, dy = displacement(rho, theta)
        ax.plot(x0 + A * dx, y0 + A * dy, **kwargs)

    ax.set(
        xlim=(-DIAGRAM_L, +DIAGRAM_L),
        ylim=(-DIAGRAM_L, +DIAGRAM_L),
        aspect="equal",
        xticks=[],
        yticks=[],
    )


# Calibrate ring displacements to wavefront coefficients (meters).  A defocused
# image maps pupil rays by dp = -grad(W); adding a coefficient c of mode j to the
# intra-focal donut therefore moves a pupil ring by
#   dp = -c * grad(Z_j) / (a4 * dZ4/drho|_1)   [in donut-radius units],
# where a4 = C.DEFOCUS_Z4 is the reference defocus that sets the donut radius
# (the ray-scale constant cancels in the ratio).  Baking this factor into
# zernike_displacement means the SAME numeric coefficient drives both the
# schematic (map_circles) and the donut (sim.simulate_donut(zernikes={j: c})).
_Z4_EDGE_SLOPE = float(
    gzk.Zernike(np.array([0.0, 0.0, 0.0, 0.0, 1.0]), R_outer=1.0, R_inner=C.EPS_RUBIN)
    .gradX(1.0, 0.0)
)
RING_SCALE = 1.0 / (C.DEFOCUS_Z4 * _Z4_EDGE_SLOPE)


def zernike_displacement(j, eps=C.EPS_RUBIN):
    """Return the intra-focal ring displacement for annular Noll mode ``j``.

    A defocused image maps pupil rays by ``dp = -grad(W)`` (see the paper's
    "Aberrations in Defocused Images" section, intra-focal side).  Here ``W`` is
    the annular Zernike ``Z_j`` on the unit pupil with central obscuration
    ``eps`` -- the *same* wavefront family danish uses to render the donut -- and
    the displacement is scaled by ``RING_SCALE`` so that the amplitude passed to
    ``map_circles`` is the wavefront coefficient in meters.  Thus one identical
    coefficient drives both the schematic ring diagram and the simulated donut,
    with no separately hand-derived gradient.  Tilt (Z2/Z3) is no different: its
    constant gradient makes the displacement a uniform shift.

    Parameters
    ----------
    j : int
        Noll index of the annular Zernike mode.
    eps : float, optional
        Fractional central obscuration (inner / outer pupil radius).

    Returns
    -------
    callable
        ``displacement(rho, theta) -> (dx, dy)`` in donut-radius units per meter
        of Zernike coefficient.
    """
    coef = np.zeros(j + 1)
    coef[j] = 1.0
    zern = gzk.Zernike(coef, R_outer=1.0, R_inner=eps)
    grad_x, grad_y = zern.gradX, zern.gradY

    def displacement(rho, theta):
        x = rho * np.cos(theta)
        y = rho * np.sin(theta)
        return -RING_SCALE * grad_x(x, y), -RING_SCALE * grad_y(x, y)

    return displacement

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.8), constrained_layout=True, dpi=150)

axes[0, 0].set_title("Defocus\n$\\nu = 0,\\, m=0$")
axes[0, 1].set_title("Tilt\n$\\nu = 0,\\, m=1$")

for col, (j, coeff) in enumerate([(4, 1e-5), (2, 3e-5)]):
    map_circles(coeff, zernike_displacement(j), ax=axes[0, col])
    show_donut(
        axes[1, col], sim.simulate_donut(zernikes={j: coeff}, defocal_type="intra")
    )

fig.savefig(C.FIGDIR / "aberrations_defocus_tilt.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(7, 3.8), constrained_layout=True, dpi=120)

axes[0, 0].set_title("Astigmatism\n$\\nu = 0,\\, m=2$")
axes[0, 1].set_title("Trefoil\n$\\nu = 0,\\, m=3$")
axes[0, 2].set_title("Quadrafoil\n$\\nu = 0,\\, m=4$")
axes[0, 3].set_title("Pentafoil\n$\\nu = 0,\\, m=5$")

for col, (j, coeff) in enumerate([(6, 7e-6), (10, 2e-6), (14, 1e-6), (20, 1e-6)]):
    map_circles(coeff, zernike_displacement(j), ax=axes[0, col])
    show_donut(
        axes[1, col], sim.simulate_donut(zernikes={j: coeff}, defocal_type="intra")
    )

fig.savefig(C.FIGDIR / "aberrations_astig_mfoil.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.8), constrained_layout=True, dpi=150)

axes[0, 0].set_title("Spherical\n$\\nu = 1,\\, m=0$")
axes[0, 1].set_title("Coma\n$\\nu = 1,\\, m=1$")

for col, (j, coeff) in enumerate([(11, 3e-7), (8, 2e-6)]):
    map_circles(coeff, zernike_displacement(j), ax=axes[0, col])
    show_donut(
        axes[1, col], sim.simulate_donut(zernikes={j: coeff}, defocal_type="intra")
    )

fig.savefig(C.FIGDIR / "aberrations_spherical_coma.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(7, 3.8), constrained_layout=True, dpi=120)

axes[0, 0].set_title("2nd Astigmatism\n$\\nu = 1,\\, m=2$")
axes[0, 1].set_title("2nd Trefoil\n$\\nu = 1,\\, m=3$")
axes[0, 2].set_title("2nd Quadrafoil\n$\\nu = 1,\\, m=4$")
axes[0, 3].set_title("2nd Pentafoil\n$\\nu = 1,\\, m=5$")

for col, (j, coeff) in enumerate([(12, 0.75e-6), (18, 0.4e-6), (26, 0.22e-6), (34, 0.12e-6)]):
    map_circles(coeff, zernike_displacement(j), ax=axes[0, col])
    show_donut(
        axes[1, col], sim.simulate_donut(zernikes={j: coeff}, defocal_type="intra")
    )

fig.savefig(C.FIGDIR / "aberrations_2nd_astig_mfoil.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.8), constrained_layout=True, dpi=150)

axes[0, 0].set_title("2nd Spherical\n$\\nu = 2,\\, m=0$")
axes[0, 1].set_title("2nd Coma\n$\\nu = 2,\\, m=1$")

for col, (j, coeff) in enumerate([(22, 0.10e-6), (16, 0.2e-6)]):
    map_circles(coeff, zernike_displacement(j), ax=axes[0, col])
    show_donut(
        axes[1, col], sim.simulate_donut(zernikes={j: coeff}, defocal_type="intra")
    )

fig.savefig(C.FIGDIR / "aberrations_2nd_spherical_coma.pdf", bbox_inches="tight")